# 🛒 Store Sales Forecasting

## Notebook 1: Data Preparation

### Project Overview

This project aims to develop a machine learning solution to forecast daily sales for multiple store-product combinations. Accurate sales forecasting enables retailers to optimize inventory levels, reduce stockouts, minimize waste, and improve operational planning.

This notebook focuses on understanding the available datasets and preparing clean, analysis-ready data before exploratory data analysis (EDA) and feature engineering.

---

## Objectives

In this notebook we will:

- Understand the available datasets
- Assess data quality
- Identify missing values and inconsistencies
- Clean and prepare the data
- Export processed datasets for subsequent analysis

---

## Deliverables

By the end of this notebook we will have:

- Cleaned datasets
- Consistent date indices
- Missing values handled appropriately
- Processed data ready for exploratory analysis

# Business Problem

Retail businesses rely on accurate demand forecasts to make decisions regarding inventory planning, procurement, staffing, and promotions. Poor forecasts can result in excess inventory, lost sales, and increased operational costs.

The objective of this project is to predict daily sales for each combination of store and product family using historical sales data along with external variables such as promotions, holidays, oil prices, store information, and customer transactions.

# Project Workflow

The project follows a structured machine learning workflow.

Raw Data
    │
    ▼
Data Preparation
    │
    ▼
Exploratory Data Analysis
    │
    ▼
Feature Engineering
    │
    ▼
Model Development
    │
    ▼
Recursive Forecasting
    │
    ▼
Final Submission


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path("..")

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

FIGURES = PROJECT_ROOT / "outputs" / "figures"
REPORTS = PROJECT_ROOT / "outputs" / "reports"

train = pd.read_csv(RAW_DATA / "train.csv", parse_dates=["date"])
test = pd.read_csv(RAW_DATA / "test.csv", parse_dates=["date"])
oil = pd.read_csv(RAW_DATA / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(RAW_DATA / "holidays_events.csv", parse_dates=["date"])
stores = pd.read_csv(RAW_DATA / "stores.csv")
transactions = pd.read_csv(RAW_DATA / "transactions.csv", parse_dates=["date"])

# Dataset Overview

The project consists of six primary datasets, each providing different information required for forecasting.

| Dataset | Purpose |
|----------|---------|
| train.csv | Historical sales and promotions |
| test.csv | Future dates requiring predictions |
| stores.csv | Store metadata |
| transactions.csv | Daily customer transactions |
| oil.csv | Daily oil prices |
| holidays_events.csv | Holidays and special events |

# 1 Initial Data Inspection

Before cleaning the data, it is important to understand the structure, size, and contents of each dataset.

In this section, we inspect:

- dataset dimensions
- data types
- missing values
- sample records

This helps identify potential data quality issues before preprocessing.

## 1.1 Training Dataset

In [3]:
print(f"Train shape: {train.shape} \nTest shape : {test.shape}")

Train shape: (3000888, 6) 
Test shape : (28512, 5)


In [4]:
print(f"{train.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[us]
 2   store_nbr    int64         
 3   family       str           
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 137.4 MB
None


In [5]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [6]:
train.isnull().sum()

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

## 1.2 Test Dataset

In [7]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           28512 non-null  int64         
 1   date         28512 non-null  datetime64[us]
 2   store_nbr    28512 non-null  int64         
 3   family       28512 non-null  str           
 4   onpromotion  28512 non-null  int64         
dtypes: datetime64[us](1), int64(3), str(1)
memory usage: 1.1 MB


In [8]:
test.isnull().sum()

id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64

In [9]:
test.head()

,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


## 1.3 Oil Price Dataset

In [10]:
oil.shape

(1218, 2)

In [11]:
oil.info()

<class 'pandas.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[us]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 19.2 KB


In [12]:
oil.isnull().sum()

date           0
dcoilwtico    43
dtype: int64

In [13]:
oil.head(20)

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27


## 1.4 Holiday & Events Dataset (`holidays_events.csv`)

This dataset contains national, regional, and local holidays along with special events that may influence consumer purchasing behaviour.

Understanding its structure helps determine how holiday-related features can be engineered later in the project.

In [14]:
holidays.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[us]
 1   type         350 non-null    str           
 2   locale       350 non-null    str           
 3   locale_name  350 non-null    str           
 4   description  350 non-null    str           
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[us](1), str(4)
memory usage: 14.1 KB


In [15]:
holidays.isnull().sum()

date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

In [16]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


## 1.5 Store Metadata (`stores.csv`)

This dataset provides descriptive information about each store, including its city, state, type, and cluster.

These attributes help distinguish different stores beyond their numerical identifiers and may improve model performance.

In [17]:
stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   store_nbr  54 non-null     int64
 1   city       54 non-null     str  
 2   state      54 non-null     str  
 3   type       54 non-null     str  
 4   cluster    54 non-null     int64
dtypes: int64(2), str(3)
memory usage: 2.2 KB


In [18]:
stores.isnull().sum()

store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

In [19]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


## 1.6 Customer Transactions (`transactions.csv`)

This dataset records the daily number of customer transactions for each store.

Transaction counts act as a proxy for customer footfall and may provide additional predictive information for sales forecasting.

In [20]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          83488 non-null  datetime64[us]
 1   store_nbr     83488 non-null  int64         
 2   transactions  83488 non-null  int64         
dtypes: datetime64[us](1), int64(2)
memory usage: 1.9 MB


In [21]:
transactions.isnull().sum()

date            0
store_nbr       0
transactions    0
dtype: int64

In [22]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


# 2. Initial Inspection Summary

The initial inspection provides a high-level understanding of each dataset before any preprocessing.

### Key Observations

- The training dataset contains historical daily sales together with promotion information.
- The test dataset has the same structure as the training dataset except that sales values are absent.
- The oil price dataset contains missing observations that will require preprocessing.
- Store metadata appears complete with no missing values.
- Customer transaction data will require further inspection before being incorporated into the forecasting pipeline.
- Holiday information contains multiple event types that will require careful processing during feature engineering.

The next section focuses on assessing these data quality issues in greater detail before cleaning the datasets.

# 3. Data Quality Assessment

After understanding the structure of each dataset, the next step is to evaluate its quality.

The objective of this section is to identify issues that may affect downstream analysis or model performance before applying any preprocessing.

The assessment focuses on:

- Missing values
- Missing dates
- Duplicate records
- Data consistency
- Dataset completeness

## 3.1 Missing Value Assessment

Missing values can reduce model performance or lead to incorrect feature engineering if not handled appropriately.

This section summarizes missing values across all datasets before any cleaning is performed.

### Assessment

The inspection in section 1 shows that:

- Training and test datasets do not contain missing values.
- Store metadata is complete.
- The oil price dataset contains missing values that require further investigation.
- Holiday information contains some missing fields that are expected due to the nature of the data (for example, locale-specific descriptions).
- Transaction data appears complete for recorded observations.

The oil dataset is identified as the primary dataset requiring preprocessing before further analysis.

## 3.2 Duplicate Record Assessment

Duplicate observations can bias descriptive statistics and negatively affect model training.

Each dataset is checked for duplicate records before preprocessing.

In [23]:
datasets = {
    "Train": train,
    "Test": test,
    "Oil": oil,
    "Holidays": holidays,
    "Stores": stores,
    "Transactions": transactions,
}

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Train: 0 duplicate rows
Test: 0 duplicate rows
Oil: 0 duplicate rows
Holidays: 0 duplicate rows
Stores: 0 duplicate rows
Transactions: 0 duplicate rows


## 3.3 Date Coverage Assessment

Since this project involves time series forecasting, continuous and consistent date ranges are essential.

The date coverage of each dataset is inspected to identify missing dates before feature engineering.

In [24]:
date_table = {
    "train": train,
    "test": test,
    "oil": oil,
    "holidays": holidays,
    "transactions": transactions
}

for name, df in date_table.items():
    print(f"Date range in {name}:")
    print(f"Min {df["date"].min()}")
    print(f"Max {df["date"].max()}\n")

Date range in train:
Min 2013-01-01 00:00:00
Max 2017-08-15 00:00:00

Date range in test:
Min 2017-08-16 00:00:00
Max 2017-08-31 00:00:00

Date range in oil:
Min 2013-01-01 00:00:00
Max 2017-08-31 00:00:00

Date range in holidays:
Min 2012-03-02 00:00:00
Max 2017-12-26 00:00:00

Date range in transactions:
Min 2013-01-01 00:00:00
Max 2017-08-15 00:00:00



### Assessment

The training and test datasets cover consecutive time periods.

However, the oil dataset does not contain observations for every calendar day, which will require restoration of missing dates during preprocessing.

# 4. Data Quality Summary

The data quality assessment identified several observations that will guide the preprocessing stage.

| Observation | Action |
|-------------|--------|
| Missing oil prices | Restore missing calendar dates and interpolate values |
| Missing calendar dates | Generate a complete daily calendar |
| Holiday metadata | Process during feature engineering |
| Duplicate records | Verify none require removal |
| Training data | No immediate quality concerns |
| Store metadata | No preprocessing required |

The next section performs the required preprocessing based on these findings.

# 5. Data Preparation

Based on the findings from the data quality assessment, this section prepares each dataset for downstream analysis.

The preprocessing steps are designed to preserve the integrity of the original data while ensuring that the datasets are suitable for exploratory analysis and feature engineering.

The preparation process includes:

- Restoring missing calendar dates
- Handling missing oil prices
- Preparing transaction data
- Creating clean datasets for subsequent notebooks

## 5.1 Restore the Training Calendar

The training dataset should contain one observation for every combination of date, store, and product family.

Since stores are closed on December 25, these combinations are absent from the original dataset. Rather than leaving gaps in the time series, the missing observations are explicitly restored with zero sales and zero promotions.

Creating a complete daily calendar ensures that subsequent lag features, rolling statistics, and forecasting models operate on continuous time series.

In [25]:
# Add missing December 25 dates and set zero sales
# Create a full calendar
full_idx = pd.MultiIndex.from_product(
    [
        pd.date_range(train["date"].min(), train["date"].max()),
        train["store_nbr"].unique(),
        train["family"].unique()
    ],
    names=["date", "store_nbr", "family"]
)

# Reindex train and add missing days
train = (
    train
    .set_index(["date", "store_nbr", "family"])
    .reindex(full_idx)
    .reset_index()
)

# Fill in new rows
train["sales"] = train["sales"].fillna(0)
train["onpromotion"] = train["onpromotion"].fillna(0)

## 5.2 Restore Daily Oil Prices

The oil price dataset contains missing dates because market prices are unavailable on certain days.

To maintain a continuous daily time series, the oil calendar is expanded to cover the complete training and testing period. Missing oil prices are then estimated using linear interpolation, ensuring that downstream time-based features can be computed without interruptions.

In [26]:
# Create a complete list of train dates
train_dates = pd.date_range(
    start=train["date"].min(),   # first date in train
    end=train["date"].max(),     # last date in train
    freq="D"                     # step = 1 day
)

# Take dates that exist in oil
oil_dates = pd.Index(oil["date"].unique())

# Find train dates missing from oil
missing_oil_dates = train_dates.difference(oil_dates)

# Check expected number of train days
print("Days in train period:", len(train_dates))

# Check the number of dates in oil
print("Dates in oil:", oil["date"].nunique())

# Count oil dates that need to be restored
print("Train period dates missing from oil:", len(missing_oil_dates))

# Look at the first missing dates
missing_oil_dates[:20]

Days in train period: 1688
Dates in oil: 1218
Train period dates missing from oil: 482


DatetimeIndex(['2013-01-05', '2013-01-06', '2013-01-12', '2013-01-13',
               '2013-01-19', '2013-01-20', '2013-01-26', '2013-01-27',
               '2013-02-02', '2013-02-03', '2013-02-09', '2013-02-10',
               '2013-02-16', '2013-02-17', '2013-02-23', '2013-02-24',
               '2013-03-02', '2013-03-03', '2013-03-09', '2013-03-10'],
              dtype='datetime64[us]', freq=None)

In [27]:
# Restore missing daily oil observations
train_start = train["date"].min()
test_end = test["date"].max()

oil = (
    oil
    .set_index("date")
    .reindex(pd.date_range(train_start, test_end, freq="D"))
    .rename_axis("date")
    .sort_index()
    .reset_index()
)

# Fill missing values with interpolation
oil["dcoilwtico"] = oil["dcoilwtico"].interpolate(
    method="linear",
    limit_direction="both"
)

# Round oil price to 2 decimal places
oil["dcoilwtico"] = oil["dcoilwtico"].round(2)

# Check the result
print("Oil date range after restoration:")
print(oil["date"].min())
print(oil["date"].max())

print("Number of missing values:")
print(oil.isnull().sum())

Oil date range after restoration:
2013-01-01 00:00:00
2017-08-31 00:00:00
Number of missing values:
date          0
dcoilwtico    0
dtype: int64


## 5.3 Prepare Transaction Data

Customer transaction records are available only for dates on which stores were open.

To ensure consistency with the restored training calendar, the transaction dataset is expanded to include all store-date combinations. Missing values are then handled appropriately, producing a complete daily transaction series for every store.

In [28]:
# Restore daily transactions series for each store
# Calculate sales by store and date
store_sales = (
    train
    .groupby(["date", "store_nbr"])["sales"]
    .sum()
    .reset_index()
)

# Create a full calendar (all dates × all stores)
full_idx = pd.MultiIndex.from_product(
    [
        pd.date_range(train["date"].min(), train["date"].max(), freq="D"),
        train["store_nbr"].unique()
    ],
    names=["date", "store_nbr"]
)

# Add missing date × store pairs
store_sales = (
    store_sales
    .set_index(["date", "store_nbr"])
    .reindex(full_idx)
    .reset_index()
)

# Merge transactions with sales
transactions = (
    transactions
    .merge(store_sales, on=["date", "store_nbr"], how="outer")
    .sort_values(["store_nbr", "date"])
)

# If there are no sales, set transactions = 0
transactions.loc[transactions["sales"] == 0, "transactions"] = 0

# Remove the temporary sales column
transactions = transactions.drop(columns=["sales"])

# Interpolate remaining missing values per store
transactions["transactions"] = (
    transactions
    .groupby("store_nbr")["transactions"]
    .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
)

# Check remaining missing values
print("Missing values after processing:")
print(transactions.isna().sum())

Missing values after processing:
date            0
store_nbr       0
transactions    0
dtype: int64


In [29]:
PROCESSED_DATA = Path("../data/processed")
PROCESSED_DATA.mkdir(exist_ok=True)

train.to_csv(PROCESSED_DATA / "train.csv", index=False)
oil.to_csv(PROCESSED_DATA / "oil.csv", index=False)
transactions.to_csv(PROCESSED_DATA / "transactions.csv", index=False)

## 5.4 Preprocessing Summary

The preprocessing stage addressed the data quality issues identified earlier in the notebook.

### Completed Steps

- Restored missing training calendar observations.
- Expanded the oil price dataset to a complete daily calendar and interpolated missing values.
- Prepared a continuous transaction dataset for each store.

The resulting datasets are now suitable for exploratory analysis, feature engineering, and model development in the subsequent notebooks.

# 6. Next Steps

The datasets prepared in this notebook provide the foundation for the remainder of the project.

The next notebook focuses on Exploratory Data Analysis (EDA), where the cleaned datasets are analyzed to understand sales behavior, seasonality, promotions, transactions, holidays, and other business patterns before feature engineering and model development.